In [1]:
import numpy as np
import tensorflow
from keras.models import Model
from keras import layers

In [2]:
!kaggle datasets download -d preetviradiya/english-hindi-dataset

Dataset URL: https://www.kaggle.com/datasets/preetviradiya/english-hindi-dataset
License(s): GNU Free Documentation License 1.3
100% 13.9M/13.9M [00:01<00:00, 7.29MB/s]



In [3]:
!unzip english-hindi-dataset.zip

Archive:  english-hindi-dataset.zip
  inflating: Dataset_English_Hindi.csv  


In [4]:
import pandas as pd
data = pd.read_csv("/content/Dataset_English_Hindi.csv")

In [5]:
import re

def clean_english(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-zA-Z?.!,]+", " ", text)
    return text.strip()

def clean_hindi(text):
    text = str(text).strip()
    # Keep Hindi Devanagari characters (\u0900-\u097F) and basic punctuation
    text = re.sub(r"[^\u0900-\u097F?.!,]+", " ", text)
    return f"[start] {text.strip()} [end]"

eng_texts = data['English'].apply(clean_english).tolist()
hin_texts = data['Hindi'].apply(clean_hindi).tolist()

In [6]:
MAX_VOCAB = 15000
MAX_LEN = 30

eng_vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB, output_sequence_length=MAX_LEN
)
hin_vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_sequence_length=MAX_LEN + 1,
    standardize=None
)

eng_vectorizer.adapt(eng_texts)
hin_vectorizer.adapt(hin_texts)

In [7]:
encoder_input_data = eng_vectorizer(eng_texts)
hin_sequences = hin_vectorizer(hin_texts)

decoder_input_data = hin_sequences[:, :-1]   # Everything except [end]
decoder_target_data = hin_sequences[:, 1:]   # Everything except [start]

In [9]:
EMB_DIM = 256
LATENT_DIM = 512

eng_vocab_size = len(eng_vectorizer.get_vocabulary())
hin_vocab_size = len(hin_vectorizer.get_vocabulary())

# --- ENCODER ---
encoder_inputs = layers.Input(shape=(None,), name="encoder_inputs")
enc_emb = layers.Embedding(eng_vocab_size, EMB_DIM, mask_zero=False)(
    encoder_inputs
)

encoder_lstm = layers.LSTM(
    LATENT_DIM, return_sequences=True, return_state=True, name="encoder_lstm"
)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

# --- DECODER ---
decoder_inputs = layers.Input(shape=(None,), name="decoder_inputs")
dec_emb_layer = layers.Embedding(hin_vocab_size, EMB_DIM, mask_zero=False)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = layers.LSTM(
    LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm"
)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# --- LUONG ATTENTION LAYER ---
# 1. Swap AdditiveAttention for layers.Attention (Luong dot-product attention)
attention_layer = layers.Attention(name="luong_attention_layer")
context_vector = attention_layer([decoder_outputs, encoder_outputs])

# 2. Concatenate context vector and decoder hidden states: [c_t; h_t]
decoder_combined_context = layers.Concatenate(
    axis=-1, name="concat_layer"
)([context_vector, decoder_outputs])

# 3. (Standard Luong Paper Step) Attentional hidden state: \tilde{h}_t = tanh(W_c [c_t; h_t])
attention_dense = layers.Dense(
    LATENT_DIM, activation="tanh", name="attentional_hidden_state"
)
attentional_outputs = attention_dense(decoder_combined_context)

# Softmax Layer
decoder_dense = layers.Dense(
    hin_vocab_size, activation="softmax", name="decoder_dense"
)
decoder_outputs = decoder_dense(attentional_outputs)

# --- FULL TRAINING MODEL ---
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  3,840,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  3,840,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, None,     │  1,574,912 │ embedding_2[0][0] │
│                     │ 512), (None,      │            │                   │
│                     │ 512), (None,      │            │                   │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │  1,574,912 │ embedding_3[0][0… │
│                     │ 512), (None,      │            │ encoder_lstm[0][… │
│                     │ 512), (None,      │            │ encoder_lstm[0][… │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ luong_attention_la… │ (None, None, 512) │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_layer        │ (None, None,      │          0 │ luong_attention_… │
│ (Concatenate)       │ 1024)             │            │ decoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attentional_hidden… │ (None, None, 512) │    524,800 │ concat_layer[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None,      │  7,695,000 │ attentional_hidd… │
│ (Dense)             │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,049,624 (72.67 MB)

 Trainable params: 19,049,624 (72.67 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
BATCH_SIZE = 64
EPOCHS = 5

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2
)

Epoch 1/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 234s 138ms/step - accuracy: 0.5548 - loss: 3.1573 - val_accuracy: 0.5772 - val_loss: 2.7583
Epoch 2/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 227s 139ms/step - accuracy: 0.6088 - loss: 2.3969 - val_accuracy: 0.6259 - val_loss: 2.1927
Epoch 3/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 227s 139ms/step - accuracy: 0.6533 - loss: 1.9169 - val_accuracy: 0.6524 - val_loss: 1.9326
Epoch 4/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 226s 138ms/step - accuracy: 0.6874 - loss: 1.6160 - val_accuracy: 0.6693 - val_loss: 1.8136
Epoch 5/5
1631/1631 ━━━━━━━━━━━━━━━━━━━━ 241s 148ms/step - accuracy: 0.7164 - loss: 1.4001 - val_accuracy: 0.6792 - val_loss: 1.7606


In [18]:
import numpy as np

# 1. Standalone Encoder Model (No changes needed)
encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

# 2. Standalone Decoder Model
decoder_state_input_h = layers.Input(shape=(LATENT_DIM,))
decoder_state_input_c = layers.Input(shape=(LATENT_DIM,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

encoder_outputs_input = layers.Input(shape=(None, LATENT_DIM))

dec_emb_inf = dec_emb_layer(decoder_inputs)
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    dec_emb_inf, initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]

# Apply the trained Luong attention layer
context_vector_inf = attention_layer(
    [decoder_outputs_inf, encoder_outputs_input]
)

# 1. Match the exact concatenation order from training: [c_t; h_t]
decoder_combined_inf = layers.Concatenate(axis=-1)(
    [context_vector_inf, decoder_outputs_inf]
)

# 2. Pass through the trained attentional hidden state layer (tanh)
attentional_outputs_inf = attention_dense(decoder_combined_inf)

# 3. Pass through the final softmax layer
decoder_outputs_inf = decoder_dense(attentional_outputs_inf)

decoder_model = Model(
    [decoder_inputs, encoder_outputs_input] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf,
)

# Vocabulary lookup maps
hin_vocab = hin_vectorizer.get_vocabulary()
hin_word_to_idx = {word: idx for idx, word in enumerate(hin_vocab)}
hin_idx_to_word = {idx: word for idx, word in enumerate(hin_vocab)}


# 3. Translation Function (No changes needed)
def translate_sentence(input_sentence):
  cleaned = clean_english(input_sentence)
  input_seq = eng_vectorizer([cleaned])

  # Unpack encoder_outputs along with h and c states
  enc_outs, h, c = encoder_model.predict(input_seq, verbose=0)
  states_value = [h, c]

  # Start with the [start] token
  target_seq = np.array([[hin_word_to_idx["[start]"]]])
  decoded_words = []

  for _ in range(MAX_LEN):
    # Pass enc_outs into decoder_model at every step
    output_tokens, h, c = decoder_model.predict(
        [target_seq, enc_outs] + states_value, verbose=0
    )

    sampled_token_index = np.argmax(output_tokens[0, -1, :])
    sampled_word = hin_idx_to_word.get(sampled_token_index, "")

    if sampled_word == "[end]" or sampled_word == "":
      break

    decoded_words.append(sampled_word)

    # Update target sequence and states for the next word
    target_seq = np.array([[sampled_token_index]])
    states_value = [h, c]

  return " ".join(decoded_words)


# Example Usage:
print(translate_sentence("How are you?"))


आप कैसे हो?
